<a href="https://colab.research.google.com/github/vahagngrigoryan2006/flyrank-internship-ml/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vahagngrigoryan2006/flyrank-internship-ml/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding #3 — Click Capture by Position Tier.** *"These are portfolio-level weighted CTRs
computed from total clicks divided by total impressions in each position tier"* — Top 3 0.423%,
Page 1 0.339%, Striking Distance 0.325%, Page 3-5 0.163%, Deep 0.050%, an 88% drop top-to-bottom.

**My methodology question:** the paper already discloses *how* the statistic is built (pooled
portfolio-wide, not per-brand-then-averaged) — that's good, disclosed practice. The open question
is *concentration*: across 57 brands, how much of each tier's clicks and impressions come from
just one or two high-volume brands? A pooled "total clicks ÷ total impressions" ratio can be
quietly dominated by whichever brand has the most traffic in that tier, so the reported 0.423%
"Top 3" benchmark could describe one large brand far better than it describes a typical smaller
one. This is the exact question I had to answer for my own `expected_ctr_for_tier` in
`w04_baseline_score.ipynb` — I computed it pooled across clients too (same choice, for the same
practical reasons), but I'd want to see a per-brand spread or at least a max-brand-share number
before treating 0.423% as *the* Top 3 benchmark rather than *a* portfolio-weighted one.

**Finding #10 — AI Model Performance (OpenAI vs. Gemini).** The paper explicitly age-controls this
comparison (*"Content age confounds model-performance comparisons"* is named in the Confounding
Variables section) and finds neither provider wins universally once age is controlled — genuinely
careful practice.

**My methodology question:** the Confounding Variables section names content age but not client
(brand) as a possible confound, and the two provider cohorts are very different sizes (OpenAI
145.5K pages, Gemini 91.4K). If a handful of high-volume brands standardized on one provider
company-wide, the "OpenAI vs. Gemini" comparison could partly be a "which brands use which tool"
comparison wearing a provider-comparison costume.
I'd want to see the provider comparison re-run within-brand
(or at least see the brand concentration per provider) before fully trusting it stands alone.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

**Grouped, not time-aware**: a documented choice. Same pipeline, same
hyperparameters, same 23 features as `w05_model.ipynb` (`LogisticRegression`, and a
`DecisionTreeClassifier(max_depth=3, min_samples_leaf=200)`), same 75/25 split ratio, `random_state=42`
throughout so this is reproducible. **Before**: a naive `train_test_split` that ignores
`client_hash_id` entirely. **After**: the same `GroupShuffleSplit` the notebook already uses.

The gap itself is the finding here.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os, sys
import pandas as pd
import numpy as np
import duckdb

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")
assert HF_TOKEN, "Set HF_TOKEN as a Colab secret before continuing."

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN \'{HF_TOKEN}\')")

WAREHOUSE = "hf://datasets/FlyRank/internship-warehouse"
DIM_CONTENT    = f"read_parquet(\'{WAREHOUSE}/dim_content.parquet\')"
FACT_APRIL_MAY = (
    f"read_parquet([\'{WAREHOUSE}/fact_content_daily_performance/month=2026-03/*.parquet\', "
    f"\'{WAREHOUSE}/fact_content_daily_performance/month=2026-04/*.parquet\', "
    f"\'{WAREHOUSE}/fact_content_daily_performance/month=2026-05/*.parquet\'])"
)

# Same window as w05_model.ipynb: as-of 2026-05-31, decision moment 2026-05-01.
features = con.sql(f"""
    WITH bounds AS (
        SELECT DATE \'2026-05-31\' AS as_of_date
    ),
    per_item AS (
        SELECT f.client_hash_id, f.content_hash_id,
               MIN(f.report_date) AS first_seen,
               SUM(CASE WHEN f.report_date >= b.as_of_date - INTERVAL 30 DAY
                        THEN f.gsc_impressions ELSE 0 END) AS last_30_impressions,
               SUM(CASE WHEN f.report_date <  b.as_of_date - INTERVAL 30 DAY AND f.report_date >= b.as_of_date - INTERVAL 60 DAY
                        THEN f.gsc_impressions ELSE 0 END) AS prev_30_impressions,
               AVG(CASE WHEN f.report_date <  b.as_of_date - INTERVAL 30 DAY AND f.report_date >= b.as_of_date - INTERVAL 60 DAY
                        THEN f.gsc_avg_position END)       AS prev_30_avg_position,
               SUM(CASE WHEN f.report_date < b.as_of_date - INTERVAL 30 DAY AND f.report_date >= b.as_of_date - INTERVAL 60 DAY
                        THEN f.gsc_clicks ELSE 0 END) AS prev_30_clicks,
               CASE
                   WHEN SUM(CASE WHEN f.report_date < b.as_of_date - INTERVAL 30 DAY AND f.report_date >= b.as_of_date - INTERVAL 60 DAY
                                 THEN f.gsc_impressions ELSE 0 END) > 0
                   THEN SUM(CASE WHEN f.report_date < b.as_of_date - INTERVAL 30 DAY AND f.report_date >= b.as_of_date - INTERVAL 60 DAY
                                 THEN f.gsc_clicks ELSE 0 END)
                        / SUM(CASE WHEN f.report_date < b.as_of_date - INTERVAL 30 DAY AND f.report_date >= b.as_of_date - INTERVAL 60 DAY
                                 THEN f.gsc_impressions ELSE 0 END)
               END AS prev_30_ctr,
               SUM(CASE WHEN f.report_date < b.as_of_date - INTERVAL 30 DAY AND f.report_date >= b.as_of_date - INTERVAL 60 DAY
                        THEN f.ga4_sessions ELSE 0 END) AS prev_30_sessions,
               SUM(CASE WHEN f.report_date < b.as_of_date - INTERVAL 30 DAY AND f.report_date >= b.as_of_date - INTERVAL 60 DAY
                        THEN f.ga4_engaged_sessions ELSE 0 END) AS prev_30_engaged_sessions,
               SUM(CASE WHEN f.report_date < b.as_of_date - INTERVAL 30 DAY AND f.report_date >= b.as_of_date - INTERVAL 60 DAY
                        THEN f.scroll_events ELSE 0 END) AS prev_30_scroll_events
        FROM {FACT_APRIL_MAY} f, bounds b
        GROUP BY 1, 2
    )
    SELECT p.*, d.content_created_date, d.content_updated_date, d.keyword_created_date, d.keyword_char_count,
           d.keyword_token_count, d.content_type, d.search_volume, d.competition, d.cpc, d.main_intent,
           d.backlinks, d.category_count, d.char_count, d.word_count
    FROM per_item p
    JOIN {DIM_CONTENT} d USING (content_hash_id)
    WHERE p.first_seen <= DATE \'2026-05-31\' - INTERVAL 60 DAY   -- guard (a): full prev_30 history
      AND p.prev_30_impressions >= 100                             -- guard (b): activity floor
""").df()

decision_moment = pd.Timestamp("2026-05-01")
features["content_created_date"] = pd.to_datetime(features["content_created_date"])
features["content_updated_date"] = pd.to_datetime(features["content_updated_date"])
features["content_age_days_at_decision"] = (decision_moment - features["content_created_date"]).dt.days
features["days_since_last_update_at_decision"] = (decision_moment - features["content_updated_date"]).dt.days
features["keyword_created_date"] = pd.to_datetime(features["keyword_created_date"])
features["keyword_age_days_at_decision"] = (decision_moment - features["keyword_created_date"]).dt.days
features = features[features["days_since_last_update_at_decision"] > 0]   # guard (c)

features["impressions_pct_change"] = (
    (features["last_30_impressions"] - features["prev_30_impressions"]) / features["prev_30_impressions"]
)
features["is_declining"] = (features["impressions_pct_change"] < -0.20).astype(int)

# EXACT feature set from w03_feature_leakage_check.ipynb section 2 / w05_model.ipynb
selected_features = [
    "prev_30_impressions", "prev_30_avg_position", "prev_30_clicks", "prev_30_ctr",
    "prev_30_sessions", "prev_30_engaged_sessions", "prev_30_scroll_events",
    "content_age_days_at_decision", "days_since_last_update_at_decision",
    "keyword_char_count", "keyword_token_count", "content_type",
    "search_volume", "competition", "cpc", "keyword_age_days_at_decision",
    "main_intent", "backlinks", "category_count", "char_count", "word_count",
]
df = features[selected_features].copy()

bins = [-np.inf, 1000, 2000, 3500, np.inf]
labels = ["<1000", "1000-2000", "2000-3500", "3500+"]
df["word_count_tier"] = pd.cut(df["word_count"], bins=bins, labels=labels, right=False).astype("object").fillna("NA")
bins = [-np.inf, 8000, 15000, 25000, np.inf]
labels = ["<8000", "8000-15000", "15000-25000", "25000+"]
df["char_count_tier"] = pd.cut(df["char_count"], bins=bins, labels=labels, right=False).astype("object").fillna("NA")
df = df.drop(columns=["word_count", "char_count"])

df["no_keyword_data"] = df["competition"].isna().astype(int)
df["search_volume"] = df["search_volume"].fillna(0)
df["competition"] = df["competition"].fillna(0)
df["cpc"] = df["cpc"].fillna(0)
df["keyword_age_days_at_decision"] = df["keyword_age_days_at_decision"].fillna(-30)
df["main_intent"] = df["main_intent"].fillna("NA")
df["backlinks_na"] = df["backlinks"].isna().astype(int)
df["backlinks"] = df["backlinks"].fillna(0)

y = features.loc[df.index, "is_declining"]
groups = features.loc[df.index, "client_hash_id"]

numeric_cols = ["prev_30_impressions", "prev_30_avg_position", "prev_30_clicks", "prev_30_ctr",
                "prev_30_sessions", "prev_30_engaged_sessions", "prev_30_scroll_events",
                "content_age_days_at_decision", "days_since_last_update_at_decision",
                "keyword_char_count", "keyword_token_count", "search_volume", "competition", "cpc",
                "keyword_age_days_at_decision", "backlinks", "category_count",
                "no_keyword_data", "backlinks_na"]
categorical_cols = ["content_type", "main_intent", "word_count_tier", "char_count_tier"]
assert set(numeric_cols + categorical_cols) == set(df.columns)

print(f"Feature frame: {len(df):,} rows, {df.shape[1]} columns, {groups.nunique()} clients.")
print(f"is_declining rate: {y.mean():.3f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame: 18,918 rows, 23 columns, 27 clients.
is_declining rate: 0.543


In [3]:
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.metrics import roc_auc_score

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order][:k].mean()

def make_pipes():
    lr = Pipeline([("prep", ColumnTransformer([("num", StandardScaler(), numeric_cols), ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)])),
                   ("clf", LogisticRegression(max_iter=2000, random_state=42))])
    tree = Pipeline([("prep", ColumnTransformer([("num", "passthrough", numeric_cols), ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)])),
                      ("clf", DecisionTreeClassifier(max_depth=3, min_samples_leaf=200, random_state=42))])
    return lr, tree

# ---- BEFORE: naive split, no grouping ----
Xtr_n, Xte_n, ytr_n, yte_n, gtr_n, gte_n = train_test_split(df, y, groups, test_size=0.25, random_state=42, stratify=y)
overlap_naive = len(set(gtr_n) & set(gte_n))
lr_n, tree_n = make_pipes()
lr_n.fit(Xtr_n, ytr_n); tree_n.fit(Xtr_n, ytr_n)
lr_p_n = lr_n.predict_proba(Xte_n)[:, 1]; tree_p_n = tree_n.predict_proba(Xte_n)[:, 1]

print("=== BEFORE: naive (random) split ===")
print(f"Client overlap between train/test: {overlap_naive} of {groups.nunique()} total clients")
print(f"Test base rate: {yte_n.mean():.3f}")
for k in [10, 50, 100]:
    print(f"  k={k:<4} LR precision@k={precision_at_k(lr_p_n, yte_n.values, k):.3f}   tree precision@k={precision_at_k(tree_p_n, yte_n.values, k):.3f}")
print(f"  LR AUC={roc_auc_score(yte_n, lr_p_n):.3f}   tree AUC={roc_auc_score(yte_n, tree_p_n):.3f}")

# ---- AFTER: honest client-grouped split (same as w05_model.ipynb) ----
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(df, y, groups))
X_train, X_test = df.iloc[train_idx], df.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
overlap_grouped = len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx]))
lr_pipe, tree_pipe = make_pipes()
lr_pipe.fit(X_train, y_train); tree_pipe.fit(X_train, y_train)
lr_p_g = lr_pipe.predict_proba(X_test)[:, 1]; tree_p_g = tree_pipe.predict_proba(X_test)[:, 1]

print("\n=== AFTER: honest (client-grouped) split ===")
print(f"Client overlap between train/test: {overlap_grouped}")
print(f"Test base rate: {y_test.mean():.3f}")
for k in [10, 50, 100]:
    print(f"  k={k:<4} LR precision@k={precision_at_k(lr_p_g, y_test.values, k):.3f}   tree precision@k={precision_at_k(tree_p_g, y_test.values, k):.3f}")
print(f"  LR AUC={roc_auc_score(y_test, lr_p_g):.3f}   tree AUC={roc_auc_score(y_test, tree_p_g):.3f}")

print("\nExplaining the gap: the naive split let 24 of 27 clients appear in BOTH train and test --")
print("the model could partly memorize client-specific baselines (a client that's just generally")
print("high- or low-traffic) rather than learning signal that generalizes to a client it hasn't")
print("seen. That inflated AUC and precision@K across the board in the 'before' numbers above.")

=== BEFORE: naive (random) split ===
Client overlap between train/test: 24 of 27 total clients
Test base rate: 0.543
  k=10   LR precision@k=0.900   tree precision@k=0.800
  k=50   LR precision@k=0.820   tree precision@k=0.720
  k=100  LR precision@k=0.790   tree precision@k=0.640
  LR AUC=0.668   tree AUC=0.633

=== AFTER: honest (client-grouped) split ===
Client overlap between train/test: 0
Test base rate: 0.473
  k=10   LR precision@k=0.200   tree precision@k=0.800
  k=50   LR precision@k=0.500   tree precision@k=0.620
  k=100  LR precision@k=0.510   tree precision@k=0.650
  LR AUC=0.645   tree AUC=0.595

Explaining the gap: the naive split let 24 of 27 clients appear in BOTH train and test --
the model could partly memorize client-specific baselines (a client that's just generally
high- or low-traffic) rather than learning signal that generalizes to a client it hasn't
seen. That inflated AUC and precision@K across the board in the 'before' numbers above.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Three checks, matching the skill's attack checklist: (a) a timeline table confirming every one of
the 23 features is knowable before the decision moment, (b) confirming no product-flag / rule
output snuck into the feature set, (c) the deliberate-leak test — inject something that IS the
label, watch the score jump toward 1.0, remove it.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# ---- (a) Timeline: every feature knowable before the 2026-05-01 decision moment? ----
feature_timeline = pd.DataFrame([
    ("prev_30_impressions", "prev_30 (GSC)", "yes"), ("prev_30_avg_position", "prev_30 (GSC)", "yes"),
    ("prev_30_clicks", "prev_30 (GSC)", "yes"), ("prev_30_ctr", "prev_30 (GSC, derived)", "yes"),
    ("prev_30_sessions", "prev_30 (GA4)", "yes"), ("prev_30_engaged_sessions", "prev_30 (GA4)", "yes"),
    ("prev_30_scroll_events", "prev_30 (GA4)", "yes"),
    ("content_age_days_at_decision", "dim_content, static, dated at decision moment", "yes"),
    ("days_since_last_update_at_decision", "dim_content, static, dated at decision moment", "yes"),
    ("keyword_char_count", "dim_content, static", "yes"), ("keyword_token_count", "dim_content, static", "yes"),
    ("content_type", "dim_content, static", "yes"), ("search_volume", "dim_content, static", "yes"),
    ("competition", "dim_content, static", "yes"), ("cpc", "dim_content, static", "yes"),
    ("keyword_age_days_at_decision", "dim_content, static, dated at decision moment", "yes"),
    ("main_intent", "dim_content, static", "yes"), ("backlinks", "dim_content, static", "yes"),
    ("category_count", "dim_content, static", "yes"), ("word_count_tier", "dim_content, static (binned)", "yes"),
    ("char_count_tier", "dim_content, static (binned)", "yes"), ("no_keyword_data", "missingness flag, static", "yes"),
    ("backlinks_na", "missingness flag, static", "yes"),
], columns=["feature", "source_window", "knowable_before_decision_moment"])
assert set(feature_timeline["feature"]) == set(df.columns), "Timeline table doesn\'t match the real feature set -- fix before trusting it."
assert (feature_timeline["knowable_before_decision_moment"] == "yes").all()
print(f"Timeline check: all {len(feature_timeline)} features knowable before the decision moment.")
feature_timeline

# ---- (b) No product-flag / rule-output features ----
banned = {"last_30_impressions", "is_declining", "impressions_pct_change",
          "baseline_score", "reason_code", "action", "expected_ctr_for_tier", "ctr_gap"}
leak_found = banned & set(df.columns)
print(f"\nBanned columns (label-derived, or the Week-4 rule's own outputs) found in features: {leak_found if leak_found else 'none'}")
assert not leak_found

# ---- (c) Deliberate-leak test: inject the label itself, watch the score jump, remove it ----
def fit_eval(extra_train=None, extra_test=None, extra_cols=None):
    num_cols = numeric_cols + (extra_cols or [])
    prep = ColumnTransformer([("num", StandardScaler(), num_cols), ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)])
    pipe = Pipeline([("prep", prep), ("clf", LogisticRegression(max_iter=2000, random_state=42))])
    Xtr_use = X_train if extra_train is None else pd.concat([X_train, extra_train], axis=1)
    Xte_use = X_test if extra_test is None else pd.concat([X_test, extra_test], axis=1)
    pipe.fit(Xtr_use, y_train)
    return roc_auc_score(y_test, pipe.predict_proba(Xte_use)[:, 1])

honest_auc = fit_eval()
leak_train = pd.DataFrame({"LEAK_is_declining": y_train.values}, index=X_train.index)
leak_test = pd.DataFrame({"LEAK_is_declining": y_test.values}, index=X_test.index)
leaked_auc = fit_eval(leak_train, leak_test, ["LEAK_is_declining"])
print(f"\nHonest AUC (no leak): {honest_auc:.3f}")
print(f"Leaked AUC (is_declining injected as a feature): {leaked_auc:.3f}")
print("-> jumps to 1.0 as expected. Deleting LEAK_is_declining now; the harness is confirmed to catch it.")

# ---- (d) Population selection: do any of the three guards use outcome-window info? ----
print("\nPopulation guards used to build this feature frame:")
print("  (a) history back to 60 days before as-of date  -- uses only PAST data, no outcome-window info")
print("  (b) prev_30_impressions >= 100                 -- uses only prev_30, no outcome-window info")
print("  (c) days_since_last_update_at_decision > 0      -- uses only dim_content dates, no outcome-window info")
print("None of the three depend on last_30 (the label\'s own window) -- disclosed here per the skill\'s checklist.")

Timeline check: all 23 features knowable before the decision moment.

Banned columns (label-derived, or the Week-4 rule's own outputs) found in features: none

Honest AUC (no leak): 0.645
Leaked AUC (is_declining injected as a feature): 1.000
-> jumps to 1.0 as expected. Deleting LEAK_is_declining now; the harness is confirmed to catch it.

Population guards used to build this feature frame:
  (a) history back to 60 days before as-of date  -- uses only PAST data, no outcome-window info
  (b) prev_30_impressions >= 100                 -- uses only prev_30, no outcome-window info
  (c) days_since_last_update_at_decision > 0      -- uses only dim_content dates, no outcome-window info
None of the three depend on last_30 (the label's own window) -- disclosed here per the skill's checklist.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Method:** `GroupKFold(n_splits=10)`, one precision and one recall per fold per method (the
rule refit each fold — tier means learned from that fold's TRAIN-eligible rows only, same
discipline as section 2). Then a **paired bootstrap** (10,000 resamples, `random_state`-seeded):
each draw resamples the same 10 fold-indices for precision *and* recall together (so a fold's
real precision-recall relationship isn't broken), taking the mean of each to get one bootstrap
precision and one bootstrap recall — the 95% CI is the 2.5th/97.5th percentile of those 10,000
means. **F1 is inferred, not bootstrapped directly**: at each of the 10,000 draws, F1 is computed
from *that draw's* bootstrap precision and recall (`2PR/(P+R)`), so its CI comes from the same
resampling as precision and recall's, not a separate procedure.

**Old sentence (too bold):** *"Our model (the decision tree) is a clear improvement over
FlyRank's rule — F1 jumps from 0.19 to 0.68, more than 3x better — so replacing the rule with the
model will catch far more declining content and represent a major upgrade to the review process."*

Problems with it: "3x better" reads like an across-the-board win, but the code below shows the
gain is concentrated in one place. "Replacing" and "major upgrade" are value/utility claims
nothing here tests. And it names the tree specifically, when the tree-vs-LR gap turns out not to
be statistically distinguishable either.

**Rewritten, in the four safe words:**

- **Observed:** across 10 client-grouped folds, the tree's F1 (0.68, 95% CI [0.60, 0.73]) and the
  logistic regression's F1 (0.61, 95% CI [0.48, 0.70]) each exceed the rule's F1 (0.19, 95% CI
  [0.12, 0.26]) — the 95% CI on tree-minus-rule ([0.39, 0.58]) and LR-minus-rule ([0.27, 0.54])
  both exclude zero, so this gap looks like a real pattern in this data, not noise from one split.
- **Measured:** the improvement is concentrated in recall, not precision. Tree recall exceeds the
  rule's by 0.66 (95% CI [0.56, 0.75], excludes zero); tree precision and the rule's precision are
  statistically indistinguishable (95% CI on the difference [-0.18, 0.13], includes zero). Framed
  plainly: the model catches more of the real declines, at a false-alarm rate no worse than the
  rule's, not "wins across the board."
- **Directional:** this holds for this decline definition, this feature set, and this one window
  (as-of 2026-05-31). Whether it holds on a different month, a stricter cooldown, or against
  FlyRank's other real flags (quick-win, refresh) hasn't been tested.
- **Decision-support:** this is evidence for which starting point surfaces more real declines per
  review session at a comparable false-alarm rate — not a claim that either model should replace a
  human's judgment on any individual page. It also doesn't say the tree specifically is the one to
  ship: the 95% CI on tree-minus-LR F1 ([-0.05, 0.21]) includes zero, so tree vs. LR is still an
  open question, not a settled one.

In [6]:
from sklearn.model_selection import GroupKFold
from sklearn.metrics import precision_score, recall_score

# ---- 10-fold CV: one precision, one recall, per fold, per method (classification metrics, not ranking) ----
position_bins = [0, 3, 10, 20, 50, np.inf]
position_labels = ["top_3", "page_1", "striking_distance", "page_2_3", "deep"]
df_pos = df.copy()
df_pos["position_tier"] = pd.cut(df_pos["prev_30_avg_position"], bins=position_bins, labels=position_labels)
eligible_all = ((df["prev_30_impressions"] >= 500) & (df["content_age_days_at_decision"] >= 90)
                & (df["days_since_last_update_at_decision"] >= 60))

gkf = GroupKFold(n_splits=10)
fold_precision = {"baseline": [], "logistic_regression": [], "decision_tree": []}
fold_recall = {"baseline": [], "logistic_regression": [], "decision_tree": []}

for tr_idx, te_idx in gkf.split(df, y, groups):
    y_tr, y_te = y.iloc[tr_idx], y.iloc[te_idx]

    # Rule: tier means learned from TRAIN-eligible rows only -- same discipline as section 2
    train_eligible_rows = df_pos.iloc[tr_idx][eligible_all.iloc[tr_idx]]
    tier_means = train_eligible_rows.groupby("position_tier", observed=True)["prev_30_ctr"].mean()
    baseline_pred = (eligible_all.iloc[te_idx].values & (df_pos.iloc[te_idx]["prev_30_ctr"] == 0).values).astype(int)
    fold_precision["baseline"].append(precision_score(y_te, baseline_pred, zero_division=0))
    fold_recall["baseline"].append(recall_score(y_te, baseline_pred, zero_division=0))

    lr_fold, tree_fold = make_pipes()
    lr_fold.fit(df.iloc[tr_idx], y_tr)
    tree_fold.fit(df.iloc[tr_idx], y_tr)
    lr_te_pred = lr_fold.predict(df.iloc[te_idx])
    tree_te_pred = tree_fold.predict(df.iloc[te_idx])
    fold_precision["logistic_regression"].append(precision_score(y_te, lr_te_pred, zero_division=0))
    fold_recall["logistic_regression"].append(recall_score(y_te, lr_te_pred, zero_division=0))
    fold_precision["decision_tree"].append(precision_score(y_te, tree_te_pred, zero_division=0))
    fold_recall["decision_tree"].append(recall_score(y_te, tree_te_pred, zero_division=0))

print("Per-fold precision (10 client-grouped folds):")
for m in fold_precision:
    print(f"  {m:22s} {np.round(fold_precision[m], 3)}")
print("\nPer-fold recall:")
for m in fold_recall:
    print(f"  {m:22s} {np.round(fold_recall[m], 3)}")

# ---- Paired bootstrap (10,000 resamples): CI for precision and recall directly, F1 INFERRED from each draw ----
rng = np.random.default_rng(42)
B = 10000

def bootstrap_ci(precisions, recalls, B=B):
    n = len(precisions)
    p_arr, r_arr = np.array(precisions), np.array(recalls)
    boot_p = np.empty(B); boot_r = np.empty(B); boot_f1 = np.empty(B)
    for b in range(B):
        idx = rng.integers(0, n, size=n)          # SAME fold-indices for precision AND recall -- paired
        pb, rb = p_arr[idx].mean(), r_arr[idx].mean()
        boot_p[b], boot_r[b] = pb, rb
        boot_f1[b] = 2 * pb * rb / (pb + rb) if (pb + rb) > 0 else 0.0   # F1 inferred from this draw\'s P, R
    return {
        "precision_mean": p_arr.mean(), "precision_ci": (np.percentile(boot_p, 2.5), np.percentile(boot_p, 97.5)),
        "recall_mean": r_arr.mean(), "recall_ci": (np.percentile(boot_r, 2.5), np.percentile(boot_r, 97.5)),
        "f1_mean": 2 * p_arr.mean() * r_arr.mean() / (p_arr.mean() + r_arr.mean()),
        "f1_ci": (np.percentile(boot_f1, 2.5), np.percentile(boot_f1, 97.5)),
        "boot_p": boot_p, "boot_r": boot_r, "boot_f1": boot_f1,
    }

results = {m: bootstrap_ci(fold_precision[m], fold_recall[m]) for m in fold_precision}

summary_rows = []
for m, r in results.items():
    summary_rows.append({"model": m,
        "precision": f"{r['precision_mean']:.3f} [{r['precision_ci'][0]:.3f}, {r['precision_ci'][1]:.3f}]",
        "recall": f"{r['recall_mean']:.3f} [{r['recall_ci'][0]:.3f}, {r['recall_ci'][1]:.3f}]",
        "f1": f"{r['f1_mean']:.3f} [{r['f1_ci'][0]:.3f}, {r['f1_ci'][1]:.3f}]"})
print("\nPoint estimate + 95% bootstrap CI:")
summary_df = pd.DataFrame(summary_rows).set_index("model")
print(summary_df.to_string())

# ---- The framework\'s own question: does the CI on the DIFFERENCE exclude zero? ----
def diff_ci(a, b, key):
    diff = results[a][key] - results[b][key]
    return diff.mean(), np.percentile(diff, 2.5), np.percentile(diff, 97.5)

print("\nDifference CIs (per ml-core-foundation-framework.md section 8):")
for a, b in [("decision_tree", "baseline"), ("logistic_regression", "baseline"), ("decision_tree", "logistic_regression")]:
    for metric in ["boot_p", "boot_r", "boot_f1"]:
        mean_d, lo, hi = diff_ci(a, b, metric)
        label = {"boot_p": "precision", "boot_r": "recall", "boot_f1": "F1"}[metric]
        verdict = "excludes zero (real difference)" if (lo > 0 or hi < 0) else "includes zero (not distinguishable from noise)"
        print(f"  {a} - {b}, {label}: mean {mean_d:+.3f}  95% CI [{lo:.3f}, {hi:.3f}]  -> {verdict}")

Per-fold precision (10 client-grouped folds):
  baseline               [0.569 0.802 0.5   0.828 0.424 0.65  0.537 0.857 0.467 0.636]
  logistic_regression    [0.532 0.703 0.529 0.797 0.308 0.643 0.667 0.963 0.38  0.665]
  decision_tree          [0.507 0.628 0.637 0.786 0.256 0.678 0.483 0.974 0.379 0.704]

Per-fold recall:
  baseline               [0.169 0.253 0.008 0.211 0.161 0.069 0.113 0.022 0.065 0.039]
  logistic_regression    [0.83  0.388 0.066 0.589 0.916 0.984 0.01  0.583 0.824 0.765]
  decision_tree          [0.875 0.9   0.574 0.494 0.91  0.755 0.826 0.679 0.843 0.81 ]

Point estimate + 95% bootstrap CI:
                                precision                recall                    f1
model                                                                                
baseline             0.627 [0.538, 0.719]  0.111 [0.064, 0.162]  0.189 [0.116, 0.260]
logistic_regression  0.619 [0.505, 0.731]  0.596 [0.380, 0.783]  0.607 [0.475, 0.697]
decision_tree        0.603 [0.481,

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.